In [ ]:
# PARALLEL PREPROCESSING FRAMEWORK

import sys
from pathlib import Path

# Import from installed xeeg_kit package
# This automatically applies BLAS thread limits via xeeg_kit/_config.py
from xeeg_kit import process_subjects_parallel
from xeeg_kit.utils import log

# >>> CONFIGURATION <<<
HOME_DIR = Path.home()
CONCAT_DIR = HOME_DIR / "xtra" / "derivatives" / "eeg" / "concat"
OUTPUT_ROOT = HOME_DIR / "xtra" / "derivatives" / "eeg" / "clean"
GPS_FILE = HOME_DIR / "xtra" / "data" / "ghw280_from_egig.gpsc"

# Verify input directory exists
if not CONCAT_DIR.exists():
    raise FileNotFoundError(f"Concatenated files directory not found: {CONCAT_DIR}")

# Channel renaming map (EGI → BEL)
rename_map = {str(i): f'E{i}' for i in range(1, 281)}
rename_map['REF CZ'] = 'Cz'

# Ensure output directory exists
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Get all concatenated FIF files
fif_files = sorted(CONCAT_DIR.glob("*.fif"))
if not fif_files:
    log("No files to process. Exiting.")
    sys.exit(0)

# Filter out already-processed files
input_files = []
for fif_path in fif_files:
    parts = fif_path.stem.split('_')
    if len(parts) >= 2:
        dataset = parts[0]
        subject_id = parts[1] if parts[1].startswith('sub-') else f"sub-{parts[1]}"
        expected_output = OUTPUT_ROOT / subject_id / dataset / f"{fif_path.stem}_mkit_cleaned.fif"
        if expected_output.exists():
            continue
    input_files.append(fif_path)

if not input_files:
    log("\nAll files already processed. Nothing to do.")
    sys.exit(0)

log(f"\n{'='*60}")
log(f"Processing {len(input_files)} new files in parallel...")
log(f"{'='*60}\n")



In [ ]:
# ============================================================================
# PARALLEL PROCESSING WITH AUTOMATIC DIRECTORY STRUCTURE
# ============================================================================

results = process_subjects_parallel(
    input_files=input_files,
    output_dir=OUTPUT_ROOT,
    output_structure="parsed",
    output_suffix="_mkit_cleaned",
    n_jobs=-1,
    gpsc_file=GPS_FILE,
    channel_rename_map=rename_map,
    run_meegkit=True,
    run_icalabel=True,
    meegkit_params={
        'low_pass_filter': 100.0,
        'notch_filter_freq': 60.0,
        'mad_threshold': 4.5,
        'asr_cutoff': 2.5,
        'star_thresh': 2.0,
        'sns_neighbors': 8,
        'drop_cz': True,
        'interpolate_bads': True,
        'find_cleanest_segment_start': 0.0,
        'find_cleanest_segment_duration': 30.0
    },
    icalabel_params={
        'mad_threshold': 5.0,
        'min_amplitude_uv': 0.1,
        'n_components': 0.98,
        'random_state': 97,
        'interpolate_bads': True
    },
    verbose=True
)

# Final summary is printed automatically by process_subjects_parallel